# 04 — Two frozen prompts, six Astra stoppers, one new Wall task set

**Study:** the original generation prompt versus the same prompt plus explicit
marginal-utility framing. Three separate construction attempts per prompt; same
model, reasoning effort, architecture, features, checkpoint tools and budgets.
Each attempt ends at one weight submission. Duplicate controllers and failed
attempts are retained. All six attempts finish **before any test evaluation**.

This is a follow-up study designed after the six-episode pilot. The generator gets
no pilot scores, preferred depth, episode values, or evaluation feedback. The
new test tasks use a fresh seed range on the **same fixed Wall layout**; they are
not a broad layout-generalization benchmark. Three generations and 12 tasks per
arm are a descriptive pilot, not grounds for equivalence/significance claims.

**Costs:** generation makes billable API calls. At the earlier ~8-minute full-depth
T4 runtime, all evaluations may take roughly **12–14 GPU hours** if the controllers
usually run to 30. Each method/episode saves independently. The evaluation cell
runs at most 8 new jobs at a time (roughly one hour at the cap), then you rerun it
to continue. Loading/warm-up add wall time but are excluded from planning timings.
The two baseline evaluations are shared across all controllers.

**Run order:** parameters → setup → frozen study → generation → evaluation chunks
→ summaries → ZIP export. Generation and evaluation switches default to `False`.
Never run multiple copies of this study concurrently against the same Drive folder.


In [ ]:
REPO_URL = "https://github.com/Artifex2002/wm_weight_space_experiments.git"
PROJECT_REF = "2ce606163917e5f9953a5842e61cb322c722c236"
EXPERIMENT_ID = "wall-pilot-001"  # Prior pilot's tasks.json and protocol.json only.
DRIVE_FOLDER = "/content/drive/MyDrive/wsp-stopper"
STUDY_ID = "wall-prompt-study-001"

MODEL_ID = "gpt-6-astra"
REASONING_EFFORT = "xhigh"
REPLICATES = 3
TEST_EPISODES = 12
TEST_SEED_START = 3000         # Distinct from pilot 1000 and diagnostics 2000.
TEST_PLANNER_SEED_START = 2000000
MAX_INSPECTION_CALLS = 40
MAX_TOOL_OUTPUT_CHARS = 200_000
MAX_API_TURNS = 45
MAX_OUTPUT_TOKENS_PER_TURN = 32_768
HTTP_TIMEOUT_SECONDS = 900   # Socket timeout; no billable request is automatically retried.
SCHEDULE_SEED = 71


## Setup and original byte-identical evaluator

Select a GPU **before setup**, and use the Drive account with the pilot handoff.
The notebook freezes the GPU model and software stack for the study; restore that
stack after a reset. Physical GPU/session metadata are recorded separately.
Use the original project commit: adding this notebook to a later commit does not
change `PROJECT_REF`. No extra Python files need to be committed to your repo.


In [ ]:
from pathlib import Path
import sys, subprocess, os, json, hashlib, shutil
from google.colab import drive
assert 'YOUR_' not in REPO_URL, 'Set REPO_URL to your repository URL'
assert '/' not in EXPERIMENT_ID and '..' not in EXPERIMENT_ID
# For a private repo, clone it manually using your preferred credentials first;
# never embed a GitHub token in REPO_URL or save one in notebook output.
drive.mount('/content/drive')
PROJECT = Path('/content/wsp-stopper')
if not (PROJECT/'.git').exists():
    subprocess.run(['git','clone',REPO_URL,str(PROJECT)],check=True)
else:
    assert subprocess.check_output(['git','-C',str(PROJECT),'remote','get-url','origin'],text=True).strip()==REPO_URL
subprocess.run(['git','-C',str(PROJECT),'checkout',PROJECT_REF],check=True)
os.chdir(PROJECT)
PROJECT_COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
required=['core.py','cli.py','agent.py','packet.py','inspection.py','wall_setup.py','cem_hook.py','upstream_patch.py']
assert all((PROJECT/'src/wsp_stopper'/f).exists() for f in required), 'Commit the repaired complete starter first'
# Retain Colab's preinstalled matched torch/torchvision pair; do not install the
# upstream all-simulator pyproject (which declares Python 3.10 only).
subprocess.run([sys.executable,'-m','pip','install','--quiet',
    'einops==0.8.1','tensordict==0.9.1','gym==0.23.1','omegaconf==2.3.0',
    'PyYAML>=6,<7','scipy','matplotlib','tqdm','pandas'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.', '--no-deps','--quiet'],check=True)
sys.path.insert(0,str(PROJECT/'src'))
import torch, torchvision, tensordict
assert tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) >= (2,7)
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU, then rerun'
print('Python:',sys.version.split()[0], 'torch:',torch.__version__,'vision:',torchvision.__version__)
print('GPU:',torch.cuda.get_device_name(0),'project:',PROJECT_COMMIT)
EXPERIMENT = Path(DRIVE_FOLDER)/EXPERIMENT_ID
EXPERIMENT.mkdir(parents=True,exist_ok=True)
WORK = PROJECT/'assets/wall-colab-v1'
BASE_PACKET = PROJECT/'packets/wall-colab-base-v1'


In [ ]:
# This exact helper is embedded in both notebooks: no extra repo files required.
RUNTIME = Path('/content/wsp_wall_notebook_runtime.py')
RUNTIME_SOURCE = '"""Shared Colab runtime. Embedded byte-identically in both notebooks.\nRuns pinned upstream model/planner classes with a small dependency surface.\nNo evaluation files are included in the generator\'s allowlisted packet.\n"""\nimport argparse\nimport ast\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport types\nfrom typing import Callable, List, Union\nfrom abc import ABC, abstractmethod\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom einops import rearrange, repeat\nfrom tensordict import TensorDict\nimport yaml\nfrom wsp_stopper.core import digest, write_json\n\n\ndef definitions(path, names, namespace=None):\n    """Execute only named classes from pinned trusted source, no unrelated imports.\n    This keeps their method bodies verbatim and avoids simulator/training imports.\n    """\n    tree = ast.parse(Path(path).read_text())\n    selected = [n for n in tree.body if isinstance(n, ast.ClassDef) and n.name in names]\n    if {n.name for n in selected} != set(names):\n        raise ValueError(f\'Missing classes in {path}\')\n    env = dict(globals())\n    if namespace: env.update(namespace)\n    env[\'__name__\'] = \'wsp_notebook_upstream\'\n    exec(compile(ast.Module(body=selected,type_ignores=[]),str(path),\'exec\'),env)\n    return [env[name] for name in names]\n\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n\n\ndef sync():\n    if torch.cuda.is_available(): torch.cuda.synchronize()\n\n\ndef configure(work):\n    work=Path(work).resolve()\n    up=work/\'jepa-wms\'\n    sys.path.insert(0,str(up))\n    os.environ[\'WSP_DINOV2_SOURCE\']=str(work/\'dinov2\')\n    os.environ[\'WSP_DINOV2_WEIGHTS\']=str(work/\'encoder.pth\')\n    torch.set_num_threads(2)\n    torch.backends.cuda.matmul.allow_tf32=False\n    torch.backends.cudnn.allow_tf32=False\n    torch.backends.cudnn.benchmark=False\n    return work,up\n\n\ndef make_environment(up):\n    # Direct DotWall construction avoids unrelated mujoco/robocasa/dataset imports.\n    from evals.simu_env_planning.envs.wall_env.envs.wall import DotWall\n    source=ast.parse((up/\'evals/simu_env_planning/envs/wall_gym_wrap.py\').read_text())\n    expr=next(n.value for n in source.body if isinstance(n,ast.Assign)\n        and any(isinstance(t,ast.Name) and t.id==\'DEFAULT_CFG\' for t in n.targets))\n    from evals.simu_env_planning.envs.wall_env.data.wall import WallDatasetConfig\n    cfg=eval(compile(ast.Expression(expr),\'wall_default_cfg\',\'eval\'),{\'WallDatasetConfig\':WallDatasetConfig})\n    cfg.device=\'cpu\'\n    # Match wrapper default layout generation; generate_random_state alternates travel direction.\n    return DotWall(rng=np.random.default_rng(0),wall_config=cfg,device=\'cpu\')\n\n\ndef create_tasks(work, output, count=10, seed_start=1000):\n    _,up=configure(work)\n    if Path(output).exists(): raise ValueError(\'Task manifest exists; reuse it instead of overwriting\')\n    seed_all(123)\n    env=make_environment(up)\n    episodes=[]\n    for i in range(count):\n        s,g=env.generate_random_state(seed_start+i)\n        episodes.append(dict(id=i,environment_seed=seed_start+i,planner_seed=900000+i,\n            start=s.tolist(),goal=g.tolist(),wall_x=float(env.wall_x),door_y=float(env.hole_y)))\n    write_json(output,dict(schema=1,task=\'Wall fixed layout cross-room navigation\',episodes=episodes))\n\n\nclass Preprocessor:\n    def __init__(self, stats, device):\n        from torchvision.transforms import Normalize\n        self.mean=torch.tensor(stats[\'proprio_mean\'])\n        self.std=torch.tensor(stats[\'proprio_std\'])\n        self.transform=Normalize([.5]*3,[.5]*3)\n    def normalize_proprios(self,x): return (x-self.mean)/self.std\n\n\ndef load_model(work,device):\n    _,up=configure(work)\n    # These modules have only small torch/einops dependencies.\n    from app.plan_common.models.vit import ViTPredictor\n    from app.plan_common.models.prop_embedding import ProprioceptiveEmbedding\n    from app.plan_common.models.dino import DinoEncoder\n    from app.vjepa_wm.video_wm import VideoWM\n    cfg=yaml.safe_load((work/\'wall.eval.yaml\').read_text())\n    kwargs=cfg[\'model_kwargs\'][\'pretrain_kwargs\']\n    stats_ast=ast.parse((up/\'app/plan_common/datasets/__init__.py\').read_text())\n    stats=next(ast.literal_eval(n.value)[\'wall\'] for n in stats_ast.body if isinstance(n,ast.Assign)\n        and any(isinstance(t,ast.Name) and t.id==\'DATA_STATS\' for t in n.targets))\n    encoder=DinoEncoder(\'dinov2_vits14\',\'x_norm_patchtokens\').to(device)\n    predictor=ViTPredictor(depth=6,heads=16,mlp_dim=2048,dropout=.1,\n        num_patches=256,num_frames=4,dim=404,use_sdpa=True).to(device)\n    def embedding(dim):\n        return ProprioceptiveEmbedding(num_frames=4,tubelet_size=1,tokens_per_step=1,\n            in_chans=dim,embed_dim=10,shift_input=False,mlp_dims=[10],use_mlp=False).to(device)\n    action,prop=embedding(10),embedding(2)\n    ckpt=torch.load(work/\'wall.pth.tar\',map_location=\'cpu\',weights_only=True)\n    report={}\n    for name,model in [(\'predictor\',predictor),(\'action_encoder\',action),(\'proprio_encoder\',prop)]:\n        state={k.replace(\'module.\',\'\'):v for k,v in ckpt[name].items()}\n        state={k.replace(\'state_encoder.weight\',\'proprio_encoder.weight\').replace(\'state_encoder.bias\',\'proprio_encoder.bias\'):v for k,v in state.items()}\n        # Some upstream versions omit regenerated causal masks from state_dict.\n        msg=model.load_state_dict(state,strict=False)\n        allowed=[k for k in msg.missing_keys if k.endswith(\'.bias\') and \'transformer.layers\' in k]\n        if msg.unexpected_keys or set(msg.missing_keys)-set(allowed):\n            raise ValueError(f\'Checkpoint mismatch: {name}: {msg}\')\n        report[name]=dict(missing_regenerated_masks=allowed,unexpected=msg.unexpected_keys)\n    video=VideoWM(encoder=encoder,predictor=predictor,action_encoder=action,proprio_encoder=prop,\n        enc_type=\'dino\',pred_type=\'dino_wm\',grid_size=16,tubelet_size_enc=1,img_size=224,\n        action_dim=10,proprio_dim=2,action_tokens=0,proprio_tokens=0,use_action=True,use_proprio=True,\n        action_skip=1,frameskip=5,action_conditioning=\'feature\',proprio_encoding=\'feature\',\n        action_encoder_inpred=False,proprio_encoder_inpred=False,batchify_video=True,\n        dup_image=False,normalize_reps=False,device=device,heads={},cfgs_loss={\'proprio_loss\':True})\n    Wrapper,=definitions(up/\'app/vjepa_wm/modelcustom/simu_env_planning/vit_enc_preds.py\',[\'EncPredWM\'])\n    wrapped=Wrapper(video,10,Preprocessor(stats,device),ctxt_window=2).to(device).eval()\n    report[\'encoder\']=\'strict=True, local pinned source and tensors\'\n    return wrapped,stats,report\n\n\ndef observation(env,state,device):\n    from torchvision.transforms import Resize\n    obs,_=env.reset(location=torch.tensor(state,dtype=torch.float32))\n    image=Resize((224,224))(obs[\'visual\'])\n    return {\'visual\':image.unsqueeze(0).unsqueeze(0).to(device),\n            \'proprio\':obs[\'proprio\'].view(1,1,2).to(device)}\n\n\ndef make_planner(work,model,z_goal,iterations,population,elites,chunk,planner_seed,device):\n    up=work/\'jepa-wms\'\n    objective_file=up/\'evals/simu_env_planning/planning/planning/objectives.py\'\n    _,Objective=definitions(objective_file,[\'BaseMPCObjective\',\'ReprTargetDistMPCObjective\'])\n    planner_file=up/\'evals/simu_env_planning/planning/planning/planner.py\'\n    _,_,CEM=definitions(planner_file,[\'PlanningResult\',\'Planner\',\'CEMPlanner\'],\n        {\'NamedTuple\':__import__(\'typing\').NamedTuple,\'objectives\':SimpleNamespace(BaseMPCObjective=object)})\n    counts={\'candidate_transition_evals\':0,\'mean_transition_evals\':0,\'candidate_sequences\':0}\n    def counted_unroll(z,act_suffix=None):\n        counts[\'mean_transition_evals\']+=act_suffix.shape[0]*act_suffix.shape[1]\n        return model.unroll(z,act_suffix=act_suffix)\n    generator=torch.Generator(device=device).manual_seed(planner_seed)\n    p=CEM(counted_unroll,iterations=iterations,num_samples=population,num_elites=elites,\n        horizon=6,action_dim=10,var_scale=1,num_act_stepped=6,\n        local_generator=generator,decode_each_iteration=False,distribute_planner=False)\n    p.device=torch.device(device)\n    p.objective=Objective(None,z_goal,sum_all_diffs=False,alpha=.1)\n    # Identical microbatching for baseline and WSP. No RNG is consumed by model eval.\n    def costs(self,actions,z):\n        vals=[]\n        for j in range(0,actions.shape[1],chunk):\n            a=actions[:,j:j+chunk]\n            predicted=model.unroll(z,act_suffix=a)\n            vals.append(self.objective(predicted,a))\n            counts[\'candidate_transition_evals\']+=a.shape[0]*a.shape[1]\n            counts[\'candidate_sequences\']+=a.shape[1]\n        return torch.cat(vals)\n    p.cost_function=types.MethodType(costs,p)\n    return p,counts\n\n\ndef evaluate(work,tasks_path,out,iterations=30,population=300,elites=10,chunk=16,weights=None,allow_cpu=False):\n    if Path(out).exists(): raise ValueError(\'Result exists; use a new filename\')\n    if weights and iterations!=30: raise ValueError(\'Controller requires cap 30\')\n    if not (2<=elites<=population and chunk>=1 and 1<=iterations<=30): raise ValueError(\'Invalid budget\')\n    if not torch.cuda.is_available() and not allow_cpu: raise RuntimeError(\'Select a Colab GPU runtime\')\n    device=\'cuda:0\' if torch.cuda.is_available() else \'cpu\'\n    work,up=configure(work)\n    seed_all(123)\n    model,stats,load_report=load_model(work,device)\n    episodes=json.loads(Path(tasks_path).read_text())[\'episodes\']\n    amean=torch.tensor(stats[\'action_mean\']);astd=torch.tensor(stats[\'action_std\'])\n    if weights: os.environ[\'WSP_STOPPER_WEIGHTS\']=str(Path(weights).resolve())\n    else: os.environ.pop(\'WSP_STOPPER_WEIGHTS\',None)\n    # Warm up encoding/one rollout, without using any scored evaluation outcome.\n    seed_all(42);env=make_environment(up)\n    warm=model.encode(observation(env,[15.,20.],device))\n    with torch.inference_mode(): model.unroll(warm,torch.zeros(1,1,10,device=device))\n    sync()\n    rows=[]\n    for ep in episodes:\n        seed_all(ep[\'environment_seed\']);env=make_environment(up)\n        if float(env.wall_x)!=ep[\'wall_x\'] or float(env.hole_y)!=ep[\'door_y\']: raise ValueError(\'Wall layout mismatch\')\n        goal_obs=observation(env,ep[\'goal\'],device)\n        start_obs=observation(env,ep[\'start\'],device)\n        sync();begin=time.perf_counter()\n        with torch.inference_mode():\n            z=model.encode(start_obs);zg=model.encode(goal_obs)\n            planner,counters=make_planner(work,model,zg,iterations,population,elites,chunk,ep[\'planner_seed\'],device)\n            result=planner.plan(z)\n        sync();elapsed=time.perf_counter()-begin\n        raw=result.actions.detach().cpu().reshape(-1,2)*astd+amean\n        seed_all(ep[\'environment_seed\'])\n        env.reset(location=torch.tensor(ep[\'start\'],dtype=torch.float32))\n        positions=[ep[\'start\']]\n        any_success=False\n        for a in raw:\n            _,_,_,info=env.step(a)\n            pos=info[\'state\'].detach().cpu().numpy()\n            positions.append(pos.tolist())\n            any_success |= bool(np.linalg.norm(pos-np.asarray(ep[\'goal\']))<4.5)\n        distance=float(np.linalg.norm(np.asarray(positions[-1])-np.asarray(ep[\'goal\'])))\n        depth=int(planner.wsp_last_depth)\n        rows.append(dict(id=ep[\'id\'],success=distance<4.5,any_step_success=any_success,\n            final_distance=distance,depth=depth,planning_seconds=elapsed,\n            utility=float(distance<4.5)-.05*depth/30,\n            actions=raw.tolist(),positions=positions,stopper_trace=planner.wsp_last_trace,\n            **counters))\n        print(f"episode {ep[\'id\']}: success={distance<4.5}, depth={depth}, seconds={elapsed:.2f}",flush=True)\n        del result,planner,z,zg\n    config=dict(horizon=6,chunk=6,frameskip=5,raw_episode_steps=30,population=population,\n        elites=elites,microbatch=chunk,cap=30,proprio_weight=.1,success=\'final_position_distance < 4.5\')\n    result=dict(schema=1,tasks_sha256=digest(tasks_path),runtime_sha256=digest(__file__),\n        release_lock=json.loads(Path(\'config/wall.release.json\').read_text()),config=config,\n        fixed_iterations=iterations,weights_sha256=digest(weights) if weights else None,\n        model_load_report=load_report,torch_version=torch.__version__,\n        device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'CPU\',episodes=rows)\n    write_json(out,result)\n    return result\n\n\ndef main():\n    p=argparse.ArgumentParser();sub=p.add_subparsers(dest=\'cmd\',required=True)\n    t=sub.add_parser(\'tasks\');t.add_argument(\'--work\',required=True);t.add_argument(\'--out\',required=True)\n    t.add_argument(\'--count\',type=int,default=10);t.add_argument(\'--seed-start\',type=int,default=1000)\n    e=sub.add_parser(\'evaluate\');e.add_argument(\'--work\',required=True);e.add_argument(\'--tasks\',required=True);e.add_argument(\'--out\',required=True)\n    e.add_argument(\'--iterations\',type=int,default=30);e.add_argument(\'--population\',type=int,default=300)\n    e.add_argument(\'--elites\',type=int,default=10);e.add_argument(\'--microbatch\',type=int,default=16)\n    e.add_argument(\'--weights\');e.add_argument(\'--allow-cpu\',action=\'store_true\')\n    a=p.parse_args()\n    if a.cmd==\'tasks\': create_tasks(a.work,a.out,a.count,a.seed_start)\n    else: evaluate(a.work,a.tasks,a.out,a.iterations,a.population,a.elites,a.microbatch,a.weights,a.allow_cpu)\n\nif __name__==\'__main__\':main()\n'
RUNTIME.write_text(RUNTIME_SOURCE)
RUNTIME_SHA=hashlib.sha256(RUNTIME.read_bytes()).hexdigest()

def run_python(*args):
    subprocess.run([sys.executable,*map(str,args)],check=True,cwd=PROJECT)

from wsp_stopper.core import digest, write_json

def project_fingerprint():
    files=list((PROJECT/'src/wsp_stopper').glob('*.py'))+[PROJECT/'config/wall.release.json',PROJECT/'prompts/generate.md']
    return {str(p.relative_to(PROJECT)):digest(p) for p in sorted(files)}


### Optional: import the pilot handoff if this Drive account does not have it

Only `tasks.json` and `protocol.json` are required to pin source/planner settings
and exclude pilot episodes. If missing, set `IMPORT_HANDOFF=True` and upload the
original `wall-pilot-001-handoff.zip` or complete-results ZIP. Only those two
allowlisted members are read; existing differing files are never overwritten.
The pilot's baseline scores and old controller are not supplied to Astra.


In [ ]:
IMPORT_HANDOFF = False
if IMPORT_HANDOFF:
    import io, zipfile
    from google.colab import files
    uploaded = files.upload()
    assert len(uploaded)==1
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        members=archive.namelist()
        wanted=[f'{EXPERIMENT_ID}/{name}' for name in ['tasks.json','protocol.json']]
        assert all(members.count(name)==1 for name in wanted)
        payload={Path(name).name:archive.read(name) for name in wanted}
    assert hashlib.sha256(payload['tasks.json']).hexdigest()==json.loads(payload['protocol.json'])['tasks_sha256']
    for name,data in payload.items():
        path=EXPERIMENT/name
        if path.exists():assert path.read_bytes()==data, f'Conflicting existing {name}'
    for name,data in payload.items():
        path=EXPERIMENT/name
        if not path.exists():path.write_bytes(data)
    print('Imported reference manifest and protocol.')
assert (EXPERIMENT/'tasks.json').is_file() and (EXPERIMENT/'protocol.json').is_file(), 'Import the pilot handoff before continuing'


## Embedded study harness

The source below handles frozen prompts/settings, identical checkpoint inspection,
failed-attempt receipts, episode-level resumption, hardware records and summaries.
It never puts task values or measured results in the inspection packet. The
separate generator histories are fresh; no model seed is assumed available.
Independent attempts can produce identical weights, which remain in the results.


In [ ]:
STUDY_HELPER_SOURCE = '"""Frozen two-prompt study. No evaluation data are passed to the generation API."""\nimport ast\nimport collections\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.util\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport subprocess\nimport sys\nimport traceback\nimport urllib.error\nimport urllib.request\nimport uuid\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nfrom wsp_stopper.core import digest, validate_weights, write_json\nfrom wsp_stopper.packet import freeze, verify\nfrom wsp_stopper.agent import API, run\n\nVARIANTS = (\'original\', \'marginal_utility\')\nFRAMING = \'\'\'\\nAdditional decision framing (the objective and interface are unchanged):\nConstruct a controller that decides whether the expected benefit of another\nplanning iteration exceeds its compute penalty under the stated utility.\nOptimization convergence is one possible signal, but is not required for stopping:\na usable plan may exist while costs and actions are still changing. Conversely,\nstable optimization does not guarantee environment success. Consider all stopping\ndepths permitted by the interface, and account for uncertainty in the benefit\nof continuing. Do not assume any preferred depth or that a cost threshold is\ncalibrated to success. The numerical submission protocol is unchanged. No\nperformance feedback is available.\n\'\'\'\n\n\ndef now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef sha_text(s):\n    return hashlib.sha256(s.encode()).hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text())\n\n\ndef immutable_json(path, data):\n    path = Path(path)\n    if path.exists():\n        if read_json(path) != data:\n            raise ValueError(f\'Frozen file differs: {path}. Restore original settings or use a new STUDY_ID.\')\n    else:\n        path.parent.mkdir(parents=True, exist_ok=True)\n        write_json(path, data)\n    return path\n\n\ndef immutable_text(path, text):\n    path = Path(path)\n    if path.exists():\n        if path.read_text() != text:\n            raise ValueError(f\'Frozen text differs: {path}\')\n    else:\n        path.parent.mkdir(parents=True, exist_ok=True)\n        path.write_text(text)\n    return path\n\n\nclass TimedAPI(API):\n    """Same API protocol; longer socket timeout, no retries or header logging."""\n    def __init__(self, timeout):\n        super().__init__()\n        self.timeout = timeout\n\n    def request(self, method, resource, body=None):\n        request = urllib.request.Request(\n            \'https://api.openai.com/v1/\' + resource,\n            data=None if body is None else json.dumps(body, allow_nan=False).encode(),\n            headers={\'Authorization\': \'Bearer \'+self.key, \'Content-Type\': \'application/json\'},\n            method=method)\n        with urllib.request.urlopen(request, timeout=self.timeout) as response:\n            return json.load(response)\n\n\nclass Study:\n    def __init__(self, project, work, base_packet, runtime, root, config, helper_sha,\n                 runner=None, generator=None, hardware=None):\n        self.project, self.work, self.base_packet, self.runtime, self.root = map(\n            lambda p: Path(p).resolve(), (project, work, base_packet, runtime, root))\n        self.config = dict(config)\n        self.helper_sha = helper_sha\n        self.child_runner = runner\n        self.generator = generator or run\n        self.hardware_override = hardware\n        self.session_id = uuid.uuid4().hex\n        self.packet = self.root/\'packet\'\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.protocol_path = self.root/\'study.json\'\n        self.plan = None\n\n    def fingerprint(self):\n        # Match notebook 02\'s project_fingerprint exactly, including relative keys.\n        files = list((self.project/\'src/wsp_stopper\').glob(\'*.py\')) + [\n            self.project/\'config/wall.release.json\', self.project/\'prompts/generate.md\']\n        return {str(p.relative_to(self.project)): digest(p) for p in sorted(files)}\n\n    def hardware(self):\n        if self.hardware_override is not None:\n            return dict(self.hardware_override)\n        import torch, torchvision\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'Select a Colab GPU runtime\')\n        try:\n            uuid_info = subprocess.check_output([\'nvidia-smi\',\'--query-gpu=uuid,name,driver_version\',\n                \'--format=csv,noheader\'], text=True).strip()\n        except Exception:\n            uuid_info = \'unavailable\'\n        return dict(device=torch.cuda.get_device_name(0), torch_version=torch.__version__,\n            torchvision_version=torchvision.__version__, numpy_version=np.__version__,\n            python_version=sys.version.split()[0], cuda_version=torch.version.cuda, gpu_info=uuid_info)\n\n    def child(self, *args):\n        if self.child_runner:\n            return self.child_runner(*args)\n        command = [sys.executable,*map(str,args)]\n        directory = self.root/\'process_logs\'; directory.mkdir(exist_ok=True)\n        path = directory/(uuid.uuid4().hex+\'.txt\')\n        tail = collections.deque(maxlen=100)\n        with path.open(\'w\') as log:\n            log.write(json.dumps(command)+\'\\n\'); log.flush()\n            with subprocess.Popen(command,cwd=self.project,stdout=subprocess.PIPE,\n                                  stderr=subprocess.STDOUT,text=True,bufsize=1) as process:\n                for line in process.stdout:\n                    print(line,end=\'\',flush=True); log.write(line); log.flush(); tail.append(line)\n                code = process.wait()\n        if code:\n            raise RuntimeError(f\'Evaluation/setup exited {code}; log {path}\\n\'+\'\'.join(tail)[-6000:])\n\n    def prepare(self):\n        self.child(\'-m\',\'wsp_stopper.wall_setup\',\'prepare\',\'--work\',self.work)\n        if not self.base_packet.exists():\n            self.child(\'-m\',\'wsp_stopper.wall_setup\',\'freeze\',\'--work\',self.work,\'--out\',self.base_packet)\n        base = verify(self.base_packet)\n        if not (self.work/\'wall.eval.yaml\').is_file():\n            raise RuntimeError(\'Base packet exists but runtime YAML is missing; inspect local preparation before proceeding\')\n        from wsp_stopper.upstream_patch import patch\n        patch(self.work/\'jepa-wms\',read_json(self.project/\'config/wall.release.json\'))\n        if self.protocol_path.exists():\n            if base[\'provenance\'][\'asset_hashes\'] != self.plan[\'asset_hashes\']:\n                raise ValueError(\'Release assets differ\')\n        return base\n\n    def freeze(self, reference_tasks, reference_protocol, extra_exclusions=()):\n        """Freeze prompts, held-out tasks, sources, packet and all study settings before API use."""\n        c = self.config\n        if c[\'replicates\'] != 3:\n            raise ValueError(\'This study predeclares exactly three attempts per variant\')\n        if c[\'test_count\'] < 1 or not c[\'model_id\']:\n            raise ValueError(\'Set test_count >= 1 and a model ID\')\n        if c[\'fixed_depths\'] != [15,30]:\n            raise ValueError(\'This study compares the predeclared fixed budgets 15 and 30\')\n        for k in [\'max_calls\',\'max_chars\',\'max_turns\',\'output_tokens\',\'http_timeout\']:\n            if c[k] < 1: raise ValueError(\'Budgets must be positive\')\n        if c[\'max_turns\'] < c[\'max_calls\']+1:\n            raise ValueError(\'Leave at least one response for final submission\')\n        reference_tasks, reference_protocol = Path(reference_tasks), Path(reference_protocol)\n        reference = read_json(reference_protocol)\n        if reference[\'tasks_sha256\'] != digest(reference_tasks):\n            raise ValueError(\'Reference task manifest mismatch\')\n        commit = subprocess.check_output([\'git\',\'-C\',str(self.project),\'rev-parse\',\'HEAD\'],text=True).strip()\n        if reference[\'project_commit\'] != commit:\n            raise ValueError(f"Project commit differs: expected {reference[\'project_commit\']}, found {commit}. Set PROJECT_REF to the recorded commit.")\n        actual_files = self.fingerprint()\n        expected_files = reference[\'project_files\']\n        if expected_files != actual_files:\n            missing = sorted(set(expected_files)-set(actual_files))\n            extra = sorted(set(actual_files)-set(expected_files))\n            changed = sorted(k for k in set(expected_files)&set(actual_files) if expected_files[k]!=actual_files[k])\n            raise ValueError(f\'Source fingerprint differs: missing={missing}, extra={extra}, changed={changed}. Restore the recorded source; do not disable this check.\')\n        if reference[\'runtime_sha256\'] != digest(self.runtime):\n            raise ValueError(\'Evaluation runtime differs from notebook 02\')\n        for k in [\'population\',\'elites\',\'microbatch\']:\n            if c[k] != reference[k]: raise ValueError(f\'Planner mismatch: {k}\')\n        if reference[\'checkpoint_lock\'] != read_json(self.project/\'config/wall.release.json\'):\n            raise ValueError(\'Checkpoint release differs\')\n        prior = read_json(self.protocol_path) if self.protocol_path.exists() else None\n        if prior:\n            self.plan = prior\n            if prior[\'settings\'] != c or prior[\'helper_sha256\'] != self.helper_sha:\n                raise ValueError(\'Frozen study settings/helper differ; restore them or use a new STUDY_ID\')\n        base = self.prepare()\n        prompt_a = (self.project/\'prompts/generate.md\').read_text()\n        prompt_b = prompt_a + FRAMING\n        prompts = {VARIANTS[0]:prompt_a,VARIANTS[1]:prompt_b}\n        for name,text in prompts.items(): immutable_text(self.root/\'prompts\'/f\'{name}.txt\',text)\n        prompt_hashes = {name:sha_text(text) for name,text in prompts.items()}\n        if prior:\n            if prior[\'prompts_sha256\'] != prompt_hashes:\n                raise ValueError(\'Prompts differ\')\n        # A persistent packet guarantees every attempt sees identical bytes across VM resets.\n        if not self.packet.exists():\n            if prior: raise ValueError(\'Frozen packet missing; restore it, do not rebuild an existing study\')\n            pending = self.root/\'packet.pending\'\n            if pending.exists(): raise ValueError(\'Incomplete packet.pending exists; inspect it before cleanup\')\n            spec = dict(base[\'specification\']); spec[\'planner\']=dict(spec[\'planner\'])\n            spec[\'planner\'].update(num_samples=c[\'population\'],num_elites=c[\'elites\'],iterations=30)\n            spec[\'planner\'][\'candidate_microbatch\']=c[\'microbatch\']\n            spec.update(success_predicate=\'Final raw position L2 distance to goal < 4.5 after 30 simulator steps; no early success termination\',\n                runtime_adapter=\'Pinned upstream classes and identical notebook runtime; see supplied notebook_runtime source\',\n                runtime_sha256=digest(self.runtime),\n                validation_status=\'Pinned source/tensor inventory checked; no empirical performance or task manifest supplied\')\n            spec_path=self.work/\'prompt-study.spec.json\'; write_json(spec_path,spec)\n            sources=[self.base_packet/rel for rel in base[\'sources\'] if not rel.endswith(\'wall.spec.json\')]\n            sources += [self.runtime,spec_path]\n            freeze(self.work/\'inspection-state.pt\',\'\',sources,spec_path,\n                dict(base[\'provenance\'],notebook_runtime_sha256=digest(self.runtime)),pending)\n            verify(pending); pending.rename(self.packet)\n        packet_manifest=verify(self.packet)\n        expected_planner=dict(base[\'specification\'][\'planner\'])\n        expected_planner.update(num_samples=c[\'population\'],num_elites=c[\'elites\'],iterations=30,candidate_microbatch=c[\'microbatch\'])\n        if packet_manifest[\'specification\'][\'runtime_sha256\'] != digest(self.runtime) or packet_manifest[\'specification\'][\'planner\'] != expected_planner:\n            raise ValueError(\'Persistent packet runtime/planner differs\')\n        # Task data live outside packet/ and are never passed to the generator.\n        task_path=self.root/\'test_tasks.json\'\n        exclusions=[reference_tasks,*map(Path,extra_exclusions)]\n        excluded=[]\n        for path in exclusions:\n            if path.is_file(): excluded.extend(read_json(path)[\'episodes\'])\n        if not task_path.exists():\n            if prior: raise ValueError(\'Frozen test manifest missing\')\n            pending=self.root/\'test_tasks.pending.json\'\n            if pending.exists(): raise ValueError(\'Inspect existing pending task manifest before cleanup\')\n            module_spec=importlib.util.spec_from_file_location(\'wsp_study_runtime\',self.runtime)\n            module=importlib.util.module_from_spec(module_spec);module_spec.loader.exec_module(module)\n            module.create_tasks(self.work,pending,c[\'test_count\'],c[\'test_seed_start\'])\n            manifest=read_json(pending)\n            for i,ep in enumerate(manifest[\'episodes\']): ep[\'planner_seed\']=c[\'planner_seed_start\']+i\n            write_json(pending,manifest)\n            candidate=manifest\n        else: candidate=read_json(task_path)\n        episodes=candidate[\'episodes\']\n        if len(episodes)!=c[\'test_count\']: raise ValueError(\'Test episode count differs\')\n        if [ep[\'environment_seed\'] for ep in episodes]!=list(range(c[\'test_seed_start\'],c[\'test_seed_start\']+c[\'test_count\'])):\n            raise ValueError(\'Environment seeds differ\')\n        if [ep[\'planner_seed\'] for ep in episodes]!=list(range(c[\'planner_seed_start\'],c[\'planner_seed_start\']+c[\'test_count\'])):\n            raise ValueError(\'Planner seeds differ\')\n        def key(ep): return tuple(ep[\'start\']),tuple(ep[\'goal\']),ep[\'wall_x\'],ep[\'door_y\']\n        if len({key(e) for e in episodes})!=len(episodes): raise ValueError(\'Duplicate test tasks\')\n        for field in [\'environment_seed\',\'planner_seed\']:\n            if {ep[field] for ep in episodes}&{ep[field] for ep in excluded}: raise ValueError(f\'Test/reference {field} overlap\')\n        if {key(e) for e in episodes}&{key(e) for e in excluded}: raise ValueError(\'Test tasks overlap earlier episodes\')\n        if not task_path.exists(): pending.rename(task_path)\n        for i,ep in enumerate(episodes):\n            immutable_json(self.root/\'tasks\'/f\'episode_{i:03d}.json\',dict(candidate,episodes=[ep]))\n        attempts=[]\n        for rep in range(1,4):\n            names=VARIANTS if rep%2 else tuple(reversed(VARIANTS))\n            for name in names: attempts.append(dict(id=f\'{name}_r{rep:02d}\',variant=name,replicate=rep))\n        methods=[dict(id=f\'fixed_{k}\',depth=k,attempt=None) for k in c[\'fixed_depths\']]\n        methods += [dict(id=a[\'id\'],depth=30,attempt=a[\'id\']) for a in attempts]\n        randomizer=random.Random(c[\'schedule_seed\']); base_order=[m[\'id\'] for m in methods];randomizer.shuffle(base_order)\n        schedule=[]\n        for i,ep in enumerate(episodes):\n            order=base_order[i%len(base_order):]+base_order[:i%len(base_order)]\n            if i//len(base_order)%2: order=list(reversed(order))\n            schedule += [dict(episode_index=i,episode_id=ep[\'id\'],method=m,order=j+1) for j,m in enumerate(order)]\n        plan=dict(schema=1,study_type=\'post-pilot prompt comparison; frozen zero-feedback construction per attempt\',\n            settings=c,helper_sha256=self.helper_sha,project_commit=commit,project_files=self.fingerprint(),\n            runtime_sha256=digest(self.runtime),reference_tasks_sha256=digest(reference_tasks),\n            test_tasks_sha256=digest(task_path),test_episode_hashes=[digest(self.root/\'tasks\'/f\'episode_{i:03d}.json\') for i in range(len(episodes))],\n            prompts_sha256=prompt_hashes,packet_sha256=digest(self.packet/\'manifest.json\'),\n            evaluation_stack={k:v for k,v in self.hardware().items() if k!=\'gpu_info\'},\n            asset_hashes=base[\'provenance\'][\'asset_hashes\'],checkpoint_lock=reference[\'checkpoint_lock\'],\n            attempts=attempts,methods=methods,schedule=schedule,\n            execution=\'One fresh subprocess per method/episode; identical warm-up; timed planning excludes loading\',\n            outcomes=dict(primary=\'mean utility: success - 0.05*depth/30\',secondary=[\'success\',\'CEM iterations\',\'planning_seconds\',\'imagined transition evaluations\']),\n            missing_generation=\'Report failures separately; do not impute unexecuted episode outcomes or replace attempts\',\n            statistics=\'Per-controller and arm summaries; small descriptive pilot, no significance or equivalence claim\')\n        immutable_json(self.protocol_path,plan);self.plan=plan\n        self.check_frozen()\n        return plan\n\n    def check_frozen(self):\n        if self.plan is None: raise RuntimeError(\'Freeze study first\')\n        p=self.plan\n        if read_json(self.protocol_path)!=p or p[\'settings\']!=self.config or p[\'helper_sha256\']!=self.helper_sha:\n            raise ValueError(\'Study definition changed\')\n        if self.fingerprint()!=p[\'project_files\'] or digest(self.runtime)!=p[\'runtime_sha256\']:\n            raise ValueError(\'Runtime/project changed\')\n        if digest(self.root/\'test_tasks.json\')!=p[\'test_tasks_sha256\']: raise ValueError(\'Test tasks changed\')\n        for i,sha in enumerate(p[\'test_episode_hashes\']):\n            if digest(self.root/\'tasks\'/f\'episode_{i:03d}.json\')!=sha: raise ValueError(\'Per-episode tasks changed\')\n        if digest(self.packet/\'manifest.json\')!=p[\'packet_sha256\']: raise ValueError(\'Packet manifest changed\')\n        for name,sha in p[\'prompts_sha256\'].items():\n            if digest(self.root/\'prompts\'/f\'{name}.txt\')!=sha: raise ValueError(\'Prompt changed\')\n\n    def receipt_path(self, aid):\n        return self.root/\'attempt_receipts\'/f\'{aid}.json\'\n\n    def attempt_state(self, attempt):\n        aid=attempt[\'id\'];path=self.receipt_path(aid);out=self.root/\'generation\'/aid\n        if path.exists():\n            r=read_json(path)\n            if r[\'attempt\']!=attempt or r[\'packet_sha256\']!=self.plan[\'packet_sha256\'] or r[\'prompt_sha256\']!=self.plan[\'prompts_sha256\'][attempt[\'variant\']]:\n                raise ValueError(\'Attempt receipt identity differs\')\n            if r[\'status\']==\'accepted\':\n                if digest(out/\'weights.json\')!=r[\'weights_sha256\']: raise ValueError(\'Accepted weights changed\')\n                validate_weights(read_json(out/\'weights.json\'))\n                run_record=read_json(out/\'run.json\')\n                if run_record[\'manifest_sha256\']!=self.plan[\'packet_sha256\'] or run_record[\'prompt_sha256\']!=r[\'prompt_sha256\'] or run_record[\'model\']!=self.config[\'model_id\']:\n                    raise ValueError(\'Accepted run provenance differs\')\n                for key,setting in [(\'max_calls\',\'max_calls\'),(\'max_chars\',\'max_chars\'),(\'max_turns\',\'max_turns\'),(\'max_output_tokens_per_turn\',\'output_tokens\'),(\'reasoning_effort\',\'reasoning_effort\')]:\n                    if run_record[key]!=self.config[setting]: raise ValueError(\'Accepted run budget/reasoning differs\')\n            return r\n        if out.exists(): raise ValueError(\'Attempt directory exists without receipt; inspect it before any API call\')\n        return dict(status=\'pending\')\n\n    def status(self):\n        self.check_frozen()\n        rows=[]\n        for a in self.plan[\'attempts\']:\n            state=self.attempt_state(a)\n            row=dict(**a,**{k:v for k,v in state.items() if k in [\'status\',\'weights_sha256\',\'error\']})\n            event_path=self.root/\'generation\'/a[\'id\']/\'events.jsonl\'\n            if event_path.exists():\n                events=[]\n                for line in event_path.read_text().splitlines():\n                    try:events.append(json.loads(line))\n                    except json.JSONDecodeError:continue  # Interrupted trailing line remains in raw audit.\n                calls=[e[\'value\'] for e in events if e.get(\'kind\')==\'tool\']\n                row[\'inspection_calls\']=len(calls)\n                row[\'tensor_read_calls\']=sum(e[\'name\']==\'read_tensor\' for e in calls)\n                row[\'inspection_output_chars\']=sum(len(e.get(\'output\',\'\')) for e in calls)\n                usage=[e[\'value\'].get(\'usage\') or {} for e in events if e.get(\'kind\')==\'response\']\n                row[\'api_response_models\']=\',\'.join(sorted({e[\'value\'].get(\'model\',\'unknown\') for e in events if e.get(\'kind\')==\'response\'}))\n                row[\'api_input_tokens\']=sum(u.get(\'input_tokens\',0) for u in usage)\n                row[\'api_cached_input_tokens\']=sum((u.get(\'input_tokens_details\') or {}).get(\'cached_tokens\',0) for u in usage)\n                row[\'api_output_tokens\']=sum(u.get(\'output_tokens\',0) for u in usage)\n                row[\'api_reasoning_tokens\']=sum((u.get(\'output_tokens_details\') or {}).get(\'reasoning_tokens\',0) for u in usage)\n            rows.append(row)\n        return pd.DataFrame(rows)\n\n    def generate(self, transport=None):\n        self.check_frozen(); verify(self.packet)\n        for attempt in self.plan[\'attempts\']:\n            state=self.attempt_state(attempt)\n            if state[\'status\']==\'accepted\':\n                print(\'Reusing accepted controller:\',attempt[\'id\']);continue\n            if state[\'status\'] in [\'failed\',\'interrupted\']:\n                print(\'Keeping failed attempt, no replacement:\',attempt[\'id\']);continue\n            if state[\'status\']==\'started\':\n                state.update(status=\'interrupted\',finished_at=now(),error=\'Interrupted/unknown API outcome; not automatically retried\')\n                write_json(self.receipt_path(attempt[\'id\']),state)\n                print(\'Marked interrupted, no retry:\',attempt[\'id\']);continue\n            # Persist a receipt before making the first request. A killed session cannot silently retry.\n            r=dict(attempt=attempt,status=\'started\',started_at=now(),packet_sha256=self.plan[\'packet_sha256\'],\n                prompt_sha256=self.plan[\'prompts_sha256\'][attempt[\'variant\']])\n            immutable_json(self.receipt_path(attempt[\'id\']),r)\n            out=self.root/\'generation\'/attempt[\'id\'];out.parent.mkdir(exist_ok=True)\n            try:\n                self.generator(self.packet,self.root/\'prompts\'/f"{attempt[\'variant\']}.txt",self.config[\'model_id\'],out,\n                    max_calls=self.config[\'max_calls\'],max_chars=self.config[\'max_chars\'],\n                    max_turns=self.config[\'max_turns\'],output_tokens=self.config[\'output_tokens\'],\n                    reasoning=self.config[\'reasoning_effort\'],transport=transport)\n                weights=out/\'weights.json\';validate_weights(read_json(weights))\n                r.update(status=\'accepted\',finished_at=now(),weights_sha256=digest(weights))\n                write_json(self.receipt_path(attempt[\'id\']),r)\n            except BaseException as error:\n                r.update(status=\'interrupted\' if isinstance(error,KeyboardInterrupt) else \'failed\',\n                    finished_at=now(),error=f\'{type(error).__name__}: {error}\')\n                write_json(self.receipt_path(attempt[\'id\']),r)\n                print(\'Attempt ended without a usable controller:\',attempt[\'id\'],r[\'error\'])\n                if isinstance(error,(KeyboardInterrupt,SystemExit)) or isinstance(error,urllib.error.HTTPError) and error.code in [401,403,429]:\n                    raise\n        return self.status()\n\n    def require_settled(self):\n        self.check_frozen()\n        states={a[\'id\']:self.attempt_state(a) for a in self.plan[\'attempts\']}\n        if any(s[\'status\'] not in [\'accepted\',\'failed\',\'interrupted\'] for s in states.values()):\n            raise RuntimeError(\'Finish all six predeclared generation attempts before evaluating or revealing test results\')\n        return states\n\n    def result_path(self, index, method):\n        return self.root/\'evaluations\'/f\'episode_{index:03d}\'/f\'{method}.json\'\n\n    def validate_result(self, data, index, method, state):\n        p=self.plan;c=self.config\n        tasks=self.root/\'tasks\'/f\'episode_{index:03d}.json\'\n        expected_sha=state[\'weights_sha256\'] if method[\'attempt\'] else None\n        if data[\'tasks_sha256\']!=digest(tasks) or data[\'runtime_sha256\']!=p[\'runtime_sha256\'] or data[\'weights_sha256\']!=expected_sha:\n            raise ValueError(\'Cached evaluation identity mismatch\')\n        if data[\'device\']!=p[\'evaluation_stack\'][\'device\'] or data[\'torch_version\']!=p[\'evaluation_stack\'][\'torch_version\']:\n            raise ValueError(\'Cached GPU model/software differs from frozen study\')\n        if data[\'fixed_iterations\']!=method[\'depth\'] or data[\'release_lock\']!=p[\'checkpoint_lock\']:\n            raise ValueError(\'Cached evaluation budget/release mismatch\')\n        config=dict(horizon=6,chunk=6,frameskip=5,raw_episode_steps=30,population=c[\'population\'],\n            elites=c[\'elites\'],microbatch=c[\'microbatch\'],cap=30,proprio_weight=.1,success=\'final_position_distance < 4.5\')\n        if data[\'config\']!=config: raise ValueError(\'Cached planner configuration differs\')\n        expected=read_json(tasks)[\'episodes\']\n        if len(data[\'episodes\'])!=1 or data[\'episodes\'][0][\'id\']!=expected[0][\'id\']:\n            raise ValueError(\'Cached episode identity mismatch\')\n        ep=data[\'episodes\'][0]\n        if not 1<=ep[\'depth\']<=method[\'depth\']: raise ValueError(\'Invalid completed depth\')\n        if method[\'attempt\'] is None and ep[\'depth\']!=method[\'depth\']: raise ValueError(\'Fixed budget did not complete\')\n        if abs(ep[\'utility\']-(float(ep[\'success\'])-.05*ep[\'depth\']/30))>1e-7: raise ValueError(\'Utility mismatch\')\n        return data\n\n    def evaluate(self, max_new_jobs=8):\n        states=self.require_settled()\n        if max_new_jobs<1: raise ValueError(\'max_new_jobs must be positive\')\n        methods={m[\'id\']:m for m in self.plan[\'methods\']}\n        hw=self.hardware(); jobs=0\n        if {k:v for k,v in hw.items() if k!=\'gpu_info\'}!=self.plan[\'evaluation_stack\']:\n            raise ValueError(\'GPU model or software stack changed; restore the frozen stack before resuming\')\n        for item in self.plan[\'schedule\']:\n            method=methods[item[\'method\']];state=states[method[\'attempt\']] if method[\'attempt\'] else None\n            if state and state[\'status\']!=\'accepted\': continue\n            index=item[\'episode_index\'];result=self.result_path(index,method[\'id\'])\n            receipt=result.with_suffix(\'.receipt.json\');started=result.with_suffix(\'.started.json\')\n            if result.exists():\n                self.validate_result(read_json(result),index,method,state)\n                if not started.exists(): raise ValueError(\'Evaluation lacks execution provenance\')\n                metadata=read_json(started)\n                if read_json(result)[\'device\']!=metadata[\'hardware\'][\'device\'] or read_json(result)[\'torch_version\']!=metadata[\'hardware\'][\'torch_version\']:\n                    raise ValueError(\'Execution hardware receipt mismatch\')\n                immutable_json(receipt,dict(result_sha256=digest(result),execution=metadata))\n                continue\n            if jobs>=max_new_jobs: break\n            self.check_frozen()\n            if state: self.attempt_state(next(a for a in self.plan[\'attempts\'] if a[\'id\']==method[\'attempt\']))\n            result.parent.mkdir(parents=True,exist_ok=True)\n            metadata=dict(session_id=self.session_id,hardware=hw,started_at=now(),scheduled=item,\n                task_sha256=self.plan[\'test_episode_hashes\'][index],weights_sha256=state[\'weights_sha256\'] if state else None,\n                runtime_sha256=self.plan[\'runtime_sha256\'])\n            if started.exists():\n                # A failed evaluation may be repeated; preserve its prior execution receipt.\n                shutil.copy2(started,started.with_name(started.stem+\'.previous-\'+uuid.uuid4().hex[:8]+\'.json\'))\n            write_json(started,metadata)\n            tasks=self.root/\'tasks\'/f\'episode_{index:03d}.json\'\n            args=[self.runtime,\'evaluate\',\'--work\',self.work,\'--tasks\',tasks,\'--out\',result,\n                \'--iterations\',method[\'depth\'],\'--population\',self.config[\'population\'],\n                \'--elites\',self.config[\'elites\'],\'--microbatch\',self.config[\'microbatch\']]\n            if state: args += [\'--weights\',self.root/\'generation\'/method[\'attempt\']/\'weights.json\']\n            print(f"Episode {index+1}/{self.config[\'test_count\']} — {method[\'id\']}",flush=True)\n            self.child(*args)\n            data=self.validate_result(read_json(result),index,method,state)\n            if data[\'device\']!=hw[\'device\'] or data[\'torch_version\']!=hw[\'torch_version\']:\n                raise ValueError(\'Child hardware differs\')\n            immutable_json(receipt,dict(result_sha256=digest(result),execution=metadata))\n            if state:self.attempt_state(next(a for a in self.plan[\'attempts\'] if a[\'id\']==method[\'attempt\']))\n            jobs+=1\n        available=sum(1 for s in self.plan[\'schedule\'] if (s[\'method\'].startswith(\'fixed_\') or states[s[\'method\']][\'status\']==\'accepted\'))\n        completed=sum(1 for s in self.plan[\'schedule\'] if self.result_path(s[\'episode_index\'],s[\'method\']).is_file())\n        print(f\'Completed {completed}/{available} executable method/episode jobs; {jobs} new jobs this call.\')\n        return completed,available\n\n    def summarize(self):\n        states=self.require_settled()\n        attempt_info={a[\'id\']:a for a in self.plan[\'attempts\']}\n        rows=[]; expected=[]\n        for method in self.plan[\'methods\']:\n            state=states[method[\'attempt\']] if method[\'attempt\'] else None\n            if state and state[\'status\']!=\'accepted\': continue\n            for i in range(self.config[\'test_count\']):\n                path=self.result_path(i,method[\'id\']);expected.append(path)\n                if not path.exists():continue\n                data=self.validate_result(read_json(path),i,method,state)\n                receipt_path=path.with_suffix(\'.receipt.json\')\n                if not receipt_path.exists():raise ValueError(\'Run evaluate again to finalize result receipts\')\n                receipt=read_json(receipt_path)\n                if receipt[\'result_sha256\']!=digest(path):raise ValueError(\'Evaluation result changed\')\n                ep=data[\'episodes\'][0];info=attempt_info.get(method[\'attempt\'],{})\n                rows.append(dict(method=method[\'id\'],variant=info.get(\'variant\',\'fixed\'),replicate=info.get(\'replicate\',0),\n                    episode_index=i,episode_id=ep[\'id\'],success=bool(ep[\'success\']),depth=ep[\'depth\'],utility=ep[\'utility\'],\n                    planning_seconds=ep[\'planning_seconds\'],final_distance=ep[\'final_distance\'],\n                    transition_evals=ep[\'candidate_transition_evals\']+ep[\'mean_transition_evals\'],\n                    session_id=receipt[\'execution\'][\'session_id\'],device=data[\'device\'],torch_version=data[\'torch_version\']))\n        complete=all(p.exists() for p in expected)\n        status=self.status();status.to_csv(self.root/\'generation_status.csv\',index=False)\n        status.to_json(self.root/\'generation_status.json\',orient=\'records\',indent=2)\n        if not rows:\n            print(\'No evaluation rows yet. Generation status:\');print(status.to_string(index=False));return dict(complete=False,status=status)\n        frame=pd.DataFrame(rows);frame.to_csv(self.root/\'episode_results.csv\',index=False)\n        methods=frame.groupby(\'method\',sort=False).agg(episodes=(\'success\',\'size\'),success_rate=(\'success\',\'mean\'),\n            mean_depth=(\'depth\',\'mean\'),mean_utility=(\'utility\',\'mean\'),mean_planning_seconds=(\'planning_seconds\',\'mean\'),\n            mean_transition_evals=(\'transition_evals\',\'mean\')).reset_index()\n        # Include failed constructions as explicit missing metrics, not silently discarded controllers.\n        for a in self.plan[\'attempts\']:\n            state=states[a[\'id\']]\n            if a[\'id\'] not in set(methods.method):\n                methods=pd.concat([methods,pd.DataFrame([dict(method=a[\'id\'],episodes=0)])],ignore_index=True)\n        methods[\'generation_status\']=methods.method.map(lambda m:states[m][\'status\'] if m in states else \'baseline\')\n        methods.to_csv(self.root/\'method_summary.csv\',index=False)\n        arm_rows=[]\n        for variant in VARIANTS:\n            aa=[a for a in self.plan[\'attempts\'] if a[\'variant\']==variant]\n            valid=[a[\'id\'] for a in aa if states[a[\'id\']][\'status\']==\'accepted\']\n            hashes=[states[m][\'weights_sha256\'] for m in valid]\n            group=methods[methods.method.isin(valid) & (methods.episodes==self.config[\'test_count\'])]\n            row=dict(variant=variant,planned_attempts=3,valid_controllers=len(valid),failed_attempts=3-len(valid),\n                unique_weight_hashes=len(set(hashes)),fully_evaluated_controllers=len(group))\n            for metric in [\'success_rate\',\'mean_depth\',\'mean_utility\']:\n                row[metric+\'_mean\']=float(group[metric].mean()) if len(group) else None\n                row[metric+\'_std_across_controllers\']=float(group[metric].std(ddof=1)) if len(group)>1 else None\n            arm_rows.append(row)\n        arms=pd.DataFrame(arm_rows);arms.to_csv(self.root/\'arm_summary.csv\',index=False)\n        pair_rows=[]\n        for method in [a[\'id\'] for a in self.plan[\'attempts\'] if states[a[\'id\']][\'status\']==\'accepted\']:\n            candidate=frame[frame.method==method].set_index(\'episode_index\')\n            for baseline in [\'fixed_15\',\'fixed_30\']:\n                reference=frame[frame.method==baseline].set_index(\'episode_index\')\n                overlap=candidate.index.intersection(reference.index)\n                if not len(overlap):continue\n                b=reference.loc[overlap];a=candidate.loc[overlap]\n                matched=(a.session_id==b.session_id)&(a.device==b.device)&(a.torch_version==b.torch_version)\n                pair_rows.append(dict(method=method,baseline=baseline,paired_episodes=len(overlap),\n                    success_delta=float(a.success.astype(float).mean()-b.success.astype(float).mean()),\n                    wsp_wins=int(((~b.success)&a.success).sum()),wsp_losses=int((b.success&(~a.success)).sum()),\n                    utility_delta=float((a.utility-b.utility).mean()),iteration_saving=float(1-a.depth.sum()/b.depth.sum()),\n                    matched_session_timing_pairs=int(matched.sum()),\n                    matched_session_time_saving=float(1-a.loc[matched].planning_seconds.sum()/b.loc[matched].planning_seconds.sum()) if matched.any() else None))\n        paired=pd.DataFrame(pair_rows);paired.to_csv(self.root/\'paired_comparisons.csv\',index=False)\n        report=dict(complete=complete,planned_controllers=6,valid_controllers=int((status.status==\'accepted\').sum()),\n            test_episodes=self.config[\'test_count\'],completed_method_episode_jobs=len(frame),expected_executable_jobs=len(expected),\n            interpretation=\'Descriptive pilot. Episodes are paired, controllers share episodes, and duplicate weights are retained. No equivalence or significance claim.\',\n            timing=\'Only same-session/device/software pairs enter paired latency savings; no repeated-timing study here.\',\n            study_sha256=digest(self.protocol_path))\n        write_json(self.root/\'summary.json\',report)\n        print(\'COMPLETE\' if complete else \'PARTIAL — finish scheduled jobs before comparing arms\')\n        return dict(complete=complete,status=status,methods=methods,arms=arms,paired=paired,episodes=frame,report=report)\n\n    def export(self, directory=\'/content\'):\n        self.check_frozen()\n        stamp=datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\')\n        target=Path(directory)/f\'{self.root.name}-{stamp}.zip\'\n        # Numeric packet tensors are large; retain the full packet in Drive, export its manifest and sources.\n        selected=[]\n        for p in self.root.rglob(\'*\'):\n            if not p.is_file():continue\n            relative=p.relative_to(self.root)\n            if relative.parts[:2]==(\'packet\',\'tensors\'):continue\n            if \'pending\' in relative.parts or p.suffix not in [\'.json\',\'.jsonl\',\'.txt\',\'.csv\',\'.png\',\'.sha256\']:continue\n            selected.append(p)\n        with zipfile.ZipFile(target,\'w\',zipfile.ZIP_DEFLATED) as archive:\n            for p in sorted(selected):archive.write(p,f\'{self.root.name}/{p.relative_to(self.root).as_posix()}\')\n        with zipfile.ZipFile(target) as archive:\n            if archive.testzip() is not None or not archive.infolist():raise RuntimeError(\'Invalid/empty export\')\n        print(\'Exported\',len(selected),\'actual files;\',target.stat().st_size,\'bytes. Tensor arrays remain in Drive.\')\n        return target\n'
STUDY_HELPER_SHA = hashlib.sha256(STUDY_HELPER_SOURCE.encode()).hexdigest()
STUDY_HELPER = Path('/content/wsp_wall_prompt_study.py')
STUDY_HELPER.write_text(STUDY_HELPER_SOURCE)
import importlib.util
_helper_spec=importlib.util.spec_from_file_location('wsp_wall_prompt_study',STUDY_HELPER)
study_module=importlib.util.module_from_spec(_helper_spec)
_helper_spec.loader.exec_module(study_module)
print('Study harness SHA:',STUDY_HELPER_SHA)


## Freeze the study before generation

This downloads/prepares the pinned model, applies the required source patch,
persists the exact numerical inspection packet in Drive (about 200 MB), and
freezes both prompts, six attempt slots, new task coordinates/seeds and evaluation
order. Reopening the notebook reuses and verifies the frozen study. Changing
settings, helper code, prompts or tasks within this study is rejected.

The original prompt is copied verbatim from the pinned repo. The second only
adds a paragraph distinguishing the value of another iteration from optimizer
convergence; it does not mention the pilot's scores or a preferred depth.
No world-model rollout or API generation occurs in this cell.


In [ ]:
assert '/' not in STUDY_ID and '..' not in STUDY_ID
reference_protocol=json.loads((EXPERIMENT/'protocol.json').read_text())
STUDY_ROOT=EXPERIMENT/'studies'/STUDY_ID
config=dict(model_id=MODEL_ID,reasoning_effort=REASONING_EFFORT,replicates=REPLICATES,
    test_count=TEST_EPISODES,test_seed_start=TEST_SEED_START,planner_seed_start=TEST_PLANNER_SEED_START,
    max_calls=MAX_INSPECTION_CALLS,max_chars=MAX_TOOL_OUTPUT_CHARS,max_turns=MAX_API_TURNS,
    output_tokens=MAX_OUTPUT_TOKENS_PER_TURN,http_timeout=HTTP_TIMEOUT_SECONDS,
    fixed_depths=[15,30],schedule_seed=SCHEDULE_SEED,
    population=reference_protocol['population'],elites=reference_protocol['elites'],microbatch=reference_protocol['microbatch'])
study=study_module.Study(PROJECT,WORK,BASE_PACKET,RUNTIME,STUDY_ROOT,config,STUDY_HELPER_SHA)
# Exclude any diagnostics/new-study manifests available in this account as well.
exclusions=[]
if (EXPERIMENT/'analysis').exists():exclusions+=list((EXPERIMENT/'analysis').rglob('heldout_tasks.json'))
if (EXPERIMENT/'studies').exists():
    exclusions += [p for p in (EXPERIMENT/'studies').glob('*/test_tasks.json') if p.parent!=STUDY_ROOT]
plan=study.freeze(EXPERIMENT/'tasks.json',EXPERIMENT/'protocol.json',exclusions)
print('Study:',STUDY_ROOT)
print('Study definition SHA:',digest(STUDY_ROOT/'study.json'))
print('Frozen test episodes:',plan['settings']['test_count'])
print('Attempts:',[a['id'] for a in plan['attempts']])
print('Only difference between prompts:')
print(study_module.FRAMING)
from IPython.display import display
display(study.status())


## Generate all six controllers — billable; no evaluation feedback

Set `RUN_GENERATION=True` once ready. Add `OPENAI_API_KEY` to Colab Secrets and
grant access, or use the hidden input. The model-list check tests visibility;
actual model/effort/tool compatibility is checked by the generation requests.

Completed controllers are reused without API calls. A failed or interrupted
construction is retained and **never replaced within this study**. A socket timeout
may have an unknown server outcome; it is not automatically retried. Authentication,
quota/rate-limit errors stop the loop; fix the account issue and rerun to continue
remaining untouched attempt slots, preserving the failed slot. Do not create
replacement study IDs based on observed test scores.

All six slots must have a terminal outcome before the evaluator can run.


In [ ]:
RUN_GENERATION = False
if RUN_GENERATION:
    current=study.status()
    transport=None
    if (current.status=='pending').any():
        from google.colab import userdata
        import getpass
        try:api_key=userdata.get('OPENAI_API_KEY')
        except Exception:api_key=getpass.getpass('OpenAI API key (hidden): ')
        assert api_key
        os.environ['OPENAI_API_KEY']=api_key
        del api_key
        transport=study_module.TimedAPI(HTTP_TIMEOUT_SECONDS)
        ids={item['id'] for item in transport.request('GET','models')['data']}
        assert MODEL_ID in ids, 'Model ID is not visible to this API key'
    display(study.generate(transport=transport))
else:
    print('Skipped API generation. Set RUN_GENERATION=True when ready.')
    display(study.status())


## Evaluate in resumable chunks — GPU, no API calls

Set `RUN_EVALUATION=True`, run this cell, then rerun it until all executable jobs
are complete. Eight methods × 12 episodes = **96 jobs** if all controllers are
valid. With the default chunk size, expect about **12 cell invocations**. Failed
constructions have no executed episodes and remain explicit in the report.

The same saved episode and planner seed are used for every method. Evaluation
order is frozen and rotates across episodes. Every job uses a fresh model process
and the same warm-up; only planning is timed. Completed jobs are checked and reused.
If an evaluation—not generation—fails, fix the infrastructure and rerun the same
cell; previous execution metadata/logs are retained. Do not run concurrently.

Resuming on another physical GPU is recorded. GPU model and software must match
the frozen study. Paired timing comparisons include only episodes evaluated in
the same notebook session; compute counts are reported for all valid pairs.


In [ ]:
RUN_EVALUATION = False
MAX_NEW_JOBS_PER_RUN = 8  # Increase intentionally for a longer uninterrupted session.
if RUN_EVALUATION:
    completed,total=study.evaluate(max_new_jobs=MAX_NEW_JOBS_PER_RUN)
    print('Remaining jobs:',total-completed)
else:
    print('Skipped GPU evaluation. Finish all six generation slots first.')


## Collect every controller's results and variation across generations

This cell requires all generation slots to be settled. It can save partial
evaluation progress, but wait for **COMPLETE** before comparing prompt arms.
Failed generations remain explicit; their unexecuted task performance is not
invented. Arm means/stds are descriptive across fully evaluated valid controllers,
with valid/failed counts and duplicate-weight counts alongside them.

The primary metric is mean utility. Success, iterations, transition evaluations,
and paired comparisons to both fixed budgets are also saved. Episode pairing does
not turn shared episodes or duplicate controllers into independent evidence.
No automatic winner selection or significance test is performed.


In [ ]:
if study.status().status.isin(['accepted','failed','interrupted']).all():
    report=study.summarize()
    display(report['status'])
    if 'methods' in report:
        display(report['methods']);display(report['arms']);display(report['paired'])
        if report['complete']:
            import matplotlib.pyplot as plt
            plotted=report['methods'][report['methods'].episodes==TEST_EPISODES].set_index('method')
            fig,axes=plt.subplots(1,3,figsize=(15,4))
            for ax,column,title in zip(axes,['success_rate','mean_depth','mean_utility'],['Success rate','Mean CEM iterations','Mean utility']):
                plotted[column].plot.bar(ax=ax,title=title,rot=65)
            axes[0].set_ylim(0,1)
            fig.tight_layout();fig.savefig(STUDY_ROOT/'comparison.png',dpi=150);plt.show()
else:
    print('Finish generation before viewing test results. Current attempt status:')
    display(study.status())


## Export the complete audit and current results

Run at any stage to download nonempty actual files. The ZIP includes frozen
prompts/study definition, test manifest, generation statuses/audits/weights,
individual evaluation JSONs, execution receipts/logs, and summaries produced so
far. It includes packet sources and the tensor inventory/hashes; large numerical
packet arrays remain in Drive. API credentials are not saved by the harness.
Review model outputs and source/weight excerpts before posting the ZIP publicly.

When all jobs finish, upload this ZIP for analysis. Keep all attempts and report
that prompt variants were designed after examining the original pilot.


In [ ]:
from google.colab import files
archive_path=study.export()
files.download(str(archive_path))
